# Phase B — GPU feature extraction

Turns the balanced text corpus from **Phase A** (`train_balanced.parquet`) into the
16-D feature matrix that **Phase C** trains on. This is the only step that benefits
from a GPU: each sample is scored against `Salesforce/codegen-350M-mono` for the
surprisal features (F1–F6). On a free Kaggle/Colab T4 the full ~22k-row run takes
roughly 20–40 minutes.

**Before you start:** enable a GPU runtime.
- Kaggle: *Settings → Accelerator → GPU T4 x2* (recommended — 30 GPU-h/week, stable).
- Colab: *Runtime → Change runtime type → T4 GPU*.

`extract_features_dataset` checkpoints every 500 rows and resumes, so if the session
disconnects you can re-run the extraction cell and it continues where it left off.

## 1. Get the Deltx source and dependencies

We add `src/` to the path rather than `pip install`-ing the package, to avoid
disturbing the platform's pre-installed CUDA build of PyTorch. Replace the clone URL
with your repository (or upload the `src/` tree).

In [ ]:
from pathlib import Path

# Pull the source. Replace REPO/CHECKOUT if you work from a fork.
REPO = "https://github.com/Sheane-mario/Deltx.git"
CHECKOUT = Path("Deltx")  # clone dir name — must match the repo name's case

if CHECKOUT.is_dir():
    print(f"{CHECKOUT}/ already present — pulling latest")
    !git -C {CHECKOUT} pull --ff-only
else:
    !git clone {REPO} {CHECKOUT}

# pyarrow>=24 is required (older DLLs break torch); transformers pulls the LM.
# torch is already CUDA-enabled on Kaggle/Colab — do NOT reinstall it.
%pip install -q "pyarrow>=24" transformers pydantic pydantic-settings rich

import sys

src = (CHECKOUT / "src").resolve()
assert (src / "deltx").is_dir(), f"No deltx package under {src} — did the clone succeed?"
sys.path.insert(0, str(src))
print("Deltx source on path:", src)

## 2. Confirm the GPU and select it

`DELTX_DEVICE` must be set **before** `DeltxConfig` is constructed. `PerplexityExtractor`
reads it, moves the model to CUDA, and runs in fp16.

In [ ]:
import os
import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU visible — enable a GPU runtime before running this notebook."
)
print("GPU:", torch.cuda.get_device_name(0))

os.environ["DELTX_DEVICE"] = "cuda"

## 3. Locate the Phase A corpus

Upload `train_balanced.parquet` (produced by `scripts/build_training_set.py`).
- **Colab:** run the `files.upload()` line below.
- **Kaggle:** add it as a dataset and point `BALANCED` at
  `/kaggle/input/<your-dataset>/train_balanced.parquet`.

In [ ]:
from pathlib import Path

# Colab upload (uncomment):
# from google.colab import files; files.upload()

BALANCED = Path("train_balanced.parquet")  # adjust for Kaggle input path
assert BALANCED.is_file(), f"Upload the Phase A parquet to {BALANCED} first."

## 4. Extract the 16-D features

First run downloads CodeGen-350M (~700 MB). Rows that fail to parse or whose feature
families error out are dropped automatically. The checkpoint at `OUT` makes this
resumable — re-run this cell after a disconnect.

In [ ]:
import pandas as pd

from deltx.common.config import DeltxConfig
from deltx.detection.dataset import DatasetManager
from deltx.detection.pipeline import FeatureExtractionPipeline

config = DeltxConfig()
assert config.device == "cuda", f"Expected device=cuda, got {config.device!r}"

manager = DatasetManager(config)
pipeline = FeatureExtractionPipeline(config)

balanced = pd.read_parquet(BALANCED)
print(f"Loaded {len(balanced)} balanced rows")

OUT = Path("data/processed/train_features.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)

features = manager.extract_features_dataset(
    balanced, pipeline, output_path=OUT, checkpoint_every=500
)
print(f"Usable: {len(features)}  |  Rejected: {len(balanced) - len(features)}")

## 5. Sanity-check the feature matrix

In [ ]:
import numpy as np

from deltx.detection.models import FeatureVector

cols = FeatureVector.feature_names()
matrix = features.loc[:, cols].to_numpy(dtype=float)
print("shape:", matrix.shape)
print("all finite:", bool(np.isfinite(matrix).all()))
print(features["label"].value_counts().to_dict())

## 6. Download the result

Move `train_features.parquet` back to `data/processed/` in your local checkout, then
run **Phase C**:

```bash
poetry run python scripts/train_detector.py --holdout-model gemini
```

In [ ]:
# Colab download (uncomment):
# from google.colab import files; files.download(str(OUT))

# Kaggle: the file under /kaggle/working is saved with the notebook output;
# commit the notebook, then download train_features.parquet from the Output tab.
print("Feature matrix ready at:", OUT.resolve())